In [1]:
# =========================================================
# 0) LIBRARY IMPORTS
# =========================================================
# Core data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# Deep learning (LSTM model)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import Huber

# KPSS and ARCH tests
from statsmodels.tsa.stattools import kpss
from statsmodels.stats.diagnostic import het_arch

# Time utilities for business-day indexing
from pandas.tseries.offsets import BDay

In [2]:
# fix random seed for reproducibility
tf.random.set_seed(66)

In [3]:
# =========================================================
# 1) DATA CLEANING
# =========================================================
def clean_stock_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans raw Yahoo Finance stock data and ensures:
    - Proper header formatting
    - Correct datetime indexing
    - Numeric consistency
    - Robust return preprocessing for feature engineering

    Main goals:
    1. Standardize raw financial data
    2. Remove malformed observations
    3. Create a stabilized return series for downstream features

    Returns:
        Cleaned pandas DataFrame indexed by Date
    """

    # Work on a copy to avoid modifying the original dataframe
    df = df.copy()

    # ---------------------------------------------------------
    # FIX YAHOO FINANCE MULTI-ROW HEADER
    # ---------------------------------------------------------
    # Some Yahoo Finance exports contain:
    #
    # Row 0 -> ticker names
    # Row 1 -> actual column names
    # Row 2+ -> market data
    #
    # This block detects that structure and rebuilds
    # the dataframe with proper column names.
    if str(df.iloc[0, 0]).lower() == "ticker":
        # Reconstruct column names
        real_cols = ["Date"] + list(df.columns[1:])

        # Remove malformed header rows
        df = df.iloc[2:].copy()

        # Apply corrected headers
        df.columns = real_cols

    # ---------------------------------------------------------
    # STANDARDIZE COLUMN NAMES
    # ---------------------------------------------------------
    # Example:
    # "close" -> "Close"
    # " volume " -> "Volume"
    #
    # This avoids downstream column mismatch issues.
    df.columns = [str(c).strip().title() for c in df.columns]

    # ---------------------------------------------------------
    # DATETIME PARSING
    # ---------------------------------------------------------
    # Ensure all observations are indexed chronologically.
    if "Date" in df.columns:
        # Convert Date column safely
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    else:
        # If Date already exists as index,
        # reset index safely before parsing
        df = df.reset_index()
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Remove invalid dates
    df = df.dropna(subset=["Date"])

    # Sort observations chronologically
    df = df.sort_values("Date")

    # Set Date as dataframe index
    df = df.set_index("Date")

    # ---------------------------------------------------------
    # ENSURE NUMERIC CONSISTENCY
    # ---------------------------------------------------------
    # Financial CSV exports occasionally contain:
    # - strings
    # - malformed values
    # - missing entries
    #
    # Convert all market variables to numeric safely.
    numeric_cols = [
        "Close",
        "High",
        "Low",
        "Open",
        "Volume"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Remove rows missing critical OHLC data
    df = df.dropna(subset=["Close", "High", "Low"])

    # ---------------------------------------------------------
    # ROBUST RETURN PREPROCESSING
    # ---------------------------------------------------------
    # Compute daily log returns.
    #
    # Log returns are preferred in quantitative finance because:
    # - they are time-additive
    # - more statistically stable
    # - naturally compatible with volatility scaling
    returns = np.log(df["Close"]).diff()

    # ---------------------------------------------------------
    # WINSORIZATION OF EXTREME RETURNS
    # ---------------------------------------------------------
    # Financial time series occasionally contain:
    # - extreme jumps
    # - bad ticks
    # - abnormal market events
    #
    # Instead of removing observations entirely,
    # we softly clip only the most extreme tails.
    #
    # IMPORTANT:
    # This cleaned series is NOT the "true" return series.
    #
    # It is used only as a stabilized feature input
    # to help neural network training remain robust.
    #
    # The raw market return dynamics are still preserved
    # elsewhere in the pipeline.
    lower = returns.quantile(0.005)
    upper = returns.quantile(0.995)

    # Store winsorized returns as a dedicated feature
    df["ret_1_clean"] = returns.clip(lower, upper)

    return df

In [4]:
# =========================================================
# 2) FEATURE ENGINEERING
# =========================================================
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates predictive features for the LSTM model.

    The goal is to expose the model to multiple dimensions
    of market behavior:

    - Returns (momentum & short-term direction)
    - Volatility (risk/regime)
    - Trend (moving averages)
    - Relative positioning (overbought/oversold)
    - Volume dynamics (market participation)

    Also defines volatility-standardized forward return
    targets for multiple forecast horizons (1 to 5 days).
    """

    df = df.copy()

    # =========================================================
    # 1) RETURN-BASED FEATURES (Momentum & short-term movement)
    # =========================================================

    # 1-day log return after winsorization
    # Used as the primary return signal to reduce
    # sensitivity to extreme outliers
    df["ret_1"] = df["ret_1_clean"]

    # Raw 1-day log return (without winsorization)
    # Preserved for diagnostics and comparison purposes
    df["ret_raw"] = np.log(df["Close"]).diff()

    # 5-day cumulative log return
    # Captures short-term trend persistence
    df["ret_5"] = df["ret_1"].rolling(5).sum()

    # 10-day momentum
    # Measures medium-term directional strength
    df["momentum_10"] = df["Close"] / df["Close"].shift(10) - 1

    # =========================================================
    # 2) VOLATILITY FEATURES (Risk & regime detection)
    # =========================================================

    # Short-term realized volatility
    # Computed as rolling standard deviation of returns
    df["volatility_5"] = df["ret_1"].rolling(5).std()

    # Medium-term realized volatility
    # Used both as:
    # - predictive feature
    # - target normalization factor
    df["volatility_20"] = df["ret_1"].rolling(20).std()

    # Intraday price range relative to close price
    # Acts as an additional volatility proxy
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]

    # =========================================================
    # 3) TREND FEATURES (Market direction & structure)
    # =========================================================

    # Short-term and medium-term moving averages
    df["ma_10"] = df["Close"].rolling(10).mean()
    df["ma_20"] = df["Close"].rolling(20).mean()

    # Trend ratio:
    # > 1 → bullish short-term trend
    # < 1 → bearish short-term trend
    df["trend_ratio"] = (df["ma_10"] / (df["ma_20"] + 1e-8))

    # Distance from moving average
    # Useful for mean reversion detection
    df["price_vs_ma20"] = df["Close"] / df["ma_20"] - 1

    # =========================================================
    # 4) RELATIVE POSITIONING (Overbought / oversold)
    # =========================================================

    # Rolling statistics for z-score computation
    rolling_mean_20 = df["Close"].rolling(20).mean()
    rolling_std_20 = df["Close"].rolling(20).std()

    # Z-score relative to recent price behavior
    # Measures how far price deviates from its
    # recent statistical average
    df["zscore_20"] = (
        df["Close"] - rolling_mean_20
    ) / rolling_std_20

    # =========================================================
    # 5) VOLUME FEATURES (Market participation & anomalies)
    # =========================================================

    if "Volume" in df.columns:
        # Relative volume compared to 20-day average
        # > 1 indicates unusually high participation
        df["volume_shock"] = (
            df["Volume"] /
            (df["Volume"].rolling(20).mean() + 1e-8)
        )

        # Short-term acceleration/deceleration in volume
        df["volume_momentum"] = (
            df["Volume"] /
            df["Volume"].shift(5) - 1
        )

    else:
        # Fallback values if volume data is unavailable
        df["volume_shock"] = 0.0
        df["volume_momentum"] = 0.0

    # =========================================================
    # 6) ATR (Average True Range)
    # =========================================================

    # ATR measures market stress and price expansion
    # by combining multiple notions of trading range

    tr1 = df["High"] - df["Low"]

    tr2 = abs(df["High"] - df["Close"].shift(1))

    tr3 = abs(df["Low"] - df["Close"].shift(1))

    # True range = maximum of the three components
    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    # 14-day Average True Range
    df["atr_14"] = true_range.rolling(14).mean()

    # =========================================================
    # 7) TARGET VARIABLES (Volatility-Standardized Returns)
    # =========================================================
    #
    # The model predicts forward returns normalized by
    # current market volatility.
    #
    # This helps stabilize the learning problem by making
    # targets more comparable across:
    # - low-volatility regimes
    # - high-volatility regimes
    # - different companies
    #
    # Without standardization, the network would need to
    # simultaneously learn both:
    # - return direction/magnitude
    # - volatility scale
    #
    # During inference, predictions are converted back into
    # raw returns by multiplying by volatility_20.

    for h in range(1, 6):
        # Forward log return for horizon h
        raw_return = np.log(df["Close"].shift(-h) / df["Close"])

        # Normalize future returns by current volatility
        df[f"target_{h}d"] = (raw_return / (df["volatility_20"] + 1e-8))

    return df

In [5]:
# =========================================================
# 3) SEQUENCE CONSTRUCTION
# =========================================================
def create_sequences(features, targets, lookback=90):
    """
    Converts tabular time-series data into sequential samples
    suitable for LSTM training.

    LSTM networks expect 3D input with shape:

        (samples, timesteps, features)

    where:
    - samples   = number of training examples
    - timesteps = historical window length
    - features  = number of engineered variables

    Each generated sample contains:
    - X: a rolling window of historical observations
    - y: the aligned forward-return target vector

    Example with lookback = 90:

    X[i] contains:
        days t-89 ... t

    y[i] contains:
        forward returns for:
        t+1, t+2, ..., t+5

    Parameters
    ----------
    features : np.ndarray
        Matrix of engineered features.

    targets : np.ndarray
        Matrix containing forward-return targets
        for all forecast horizons.

    lookback : int
        Number of historical timesteps used
        as model input.

    Returns
    -------
    X : np.ndarray
        3D tensor of sequential feature windows.

    y : np.ndarray
        Target vectors aligned with each sequence.
    """

    # Containers for generated sequences
    X, y = [], []

    # ---------------------------------------------------------
    # BUILD ROLLING WINDOWS
    # ---------------------------------------------------------
    #
    # For each timestep:
    # - use the previous `lookback` observations
    # - predict the target at the current timestep
    #
    # Example:
    #
    # X[0] = observations [0:90]
    # y[0] = target at timestep 90
    #
    for i in range(lookback, len(features)):
        # Historical feature window
        X.append(features[i - lookback:i])

        # Multi-horizon target vector
        # Shape:
        # [t+1, t+2, t+3, t+4, t+5]
        y.append(targets[i])

    # Convert Python lists into NumPy arrays
    # for TensorFlow compatibility
    return np.array(X), np.array(y)

In [6]:
# =========================================================
# 4) TRAIN LSTM FORECASTER
# =========================================================
def train_lstm_forecaster(df: pd.DataFrame, lookback=90, horizon=5):
    """
    Full end-to-end training pipeline:

    Steps:
    1. Clean raw price data
    2. Generate engineered features
    3. Construct volatility-normalized multi-horizon targets
    4. Scale feature space
    5. Convert data into LSTM sequences
    6. Train neural network model
    """

    # ---------------------------------------------------------
    # DATA PREPARATION
    # ---------------------------------------------------------
    df = clean_stock_data(df)
    df = add_features(df)

    # Feature set used for prediction
    # (only observable market information, no leakage)
    feature_cols = [
        "ret_1",
        "ret_5",
        "momentum_10",
        "volatility_5",
        "volatility_20",
        "range_pct",
        "atr_14",
        "trend_ratio",
        "price_vs_ma20",
        "zscore_20",
        "volume_shock",
        "volume_momentum"
    ]

    # ---------------------------------------------------------
    # TARGET DEFINITION
    # ---------------------------------------------------------
    # Uses volatility-normalized forward returns:
    # target_hd = log_return(t → t+h) / volatility_20(t)

    train_df = df.dropna(subset=feature_cols + [f"target_{h}d" for h in range(1, 6)]).copy()

    # ---------------------------------------------------------
    # REMOVE INF / EXTREME VALUES
    # ---------------------------------------------------------
    train_df = train_df.replace([np.inf, -np.inf], np.nan)

    train_df = train_df.dropna(subset=feature_cols + [f"target_{h}d" for h in range(1, 6)])

    # Optional stability constraint for neural training
    train_df[feature_cols] = train_df[feature_cols].clip(lower=-1e6, upper=1e6)

    X_raw = train_df[feature_cols].values
    y_raw = train_df[[f"target_{h}d" for h in range(1, 6)]].values

    # ---------------------------------------------------------
    # FEATURE SCALING
    # ---------------------------------------------------------
    scaler = StandardScaler()

    # Fit ONLY on training data (prevents leakage)
    X_scaled = scaler.fit_transform(X_raw)

    # ---------------------------------------------------------
    # SEQUENCE CONSTRUCTION
    # ---------------------------------------------------------
    X, y = create_sequences(X_scaled, y_raw, lookback=lookback)

    # ---------------------------------------------------------
    # TRAIN / TEST SPLIT (time-ordered)
    # ---------------------------------------------------------
    split = int(len(X) * 0.8)

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # ---------------------------------------------------------
    # MODEL INITIALIZATION
    # ---------------------------------------------------------
    model = build_mc_lstm(input_shape=(X_train.shape[1], X_train.shape[2]))

    # Early stopping prevents overfitting on time series noise
    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

    # ---------------------------------------------------------
    # MODEL TRAINING
    # ---------------------------------------------------------
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=40,
        batch_size=32,
        callbacks=[es],
        verbose=1
    )

    return model, scaler, df, feature_cols

In [7]:
# =========================================================
# 5) MODEL ARCHITECTURE
# =========================================================
def build_mc_lstm(input_shape):
    """
    LSTM model designed for multi-horizon financial forecasting.

    Uses dropout not only for regularization, but also for
    Monte Carlo Dropout inference (uncertainty estimation
    at prediction time).
    """

    model = Sequential([
        LSTM(64, input_shape=input_shape), # Regularization + MC Dropout (active during inference if training=True)
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dropout(0.3),
        Dense(16, activation="relu"),
        Dropout(0.2),
        Dense(5) # Outputs a vector of size 5: [t+1, t+2, t+3, t+4, t+5] volatility-normalized returns
    ])

    model.compile(
        optimizer="adam",
        # Loss function choice (experimented alternatives):
        # - MAE: stable, robust to outliers (current choice)
        # - Huber: balance between MAE and MSE (often best for finance)
        # - MSE: penalizes large errors more heavily
        loss="mae",
        metrics=["mae"]
    )

    return model

In [8]:
def compute_var_cvar(pred_samples, alpha=0.05):
    """
    Computes financial risk metrics:
    - VaR (Value at Risk)
    - CVaR (Conditional Value at Risk)

    These are derived from simulated return distributions.
    """
    var = np.percentile(pred_samples, alpha * 100)
    cvar = pred_samples[pred_samples <= var].mean()
    return var, cvar

In [9]:
# =========================================================
# 6) MONTE CARLO FORECAST (FIXED MULTI-HORIZON VERSION)
# =========================================================
def mc_dropout_forecast_path(
    model,
    scaler,
    df,
    feature_cols,
    lookback=90,
    horizon=5,
    n_samples=200
):
    """
    Generates probabilistic multi-horizon forecasts using
    Monte Carlo Dropout.

    Key idea:
    - Dropout remains active at inference time (training=True)
    - Produces a distribution of forecasts per horizon
    - Enables estimation of uncertainty bands and risk metrics

    Unlike recursive forecasting:
    - all horizons are predicted in a single forward pass
    - uncertainty is estimated per horizon independently
    """

    # ---------------------------------------------------------
    # PREPARE LATEST FEATURE WINDOW
    # ---------------------------------------------------------
    df = df.replace([np.inf, -np.inf], np.nan)

    # Clip extreme values for numerical stability in inference
    df = df.clip(-1e6, 1e6)
    df = df.dropna(subset=feature_cols)
    latest_features = df[feature_cols].values[-lookback:]
    latest_scaled = scaler.transform(latest_features)

    X_input = latest_scaled.reshape(1, lookback, len(feature_cols))

    # ---------------------------------------------------------
    # MONTE CARLO DROPOUT SAMPLING
    # ---------------------------------------------------------
    # Each forward pass with training=True activates dropout,
    # producing a different stochastic prediction.
    #
    # Shape:
    # (n_samples, horizon)
    #
    # Each row = one simulated forecast path
    pred_samples = []

    for _ in range(n_samples):
        pred = model(X_input, training=True).numpy()[0]
        pred_samples.append(pred)

    pred_samples = np.array(pred_samples)

    # ---------------------------------------------------------
    # LAST OBSERVED MARKET PRICE
    # ---------------------------------------------------------
    last_price = df["Close"].iloc[-1]

    # ---------------------------------------------------------
    # FUTURE BUSINESS DATES
    # ---------------------------------------------------------
    future_dates = pd.bdate_range(start=df.index[-1] + BDay(1), periods=horizon)

    rows = []

    # ---------------------------------------------------------
    # COMPUTE FORECAST STATISTICS PER HORIZON
    # ---------------------------------------------------------
    for h in range(horizon):
        # Convert normalized predictions back to returns
        latest_vol = df["volatility_20"].iloc[-1]
        horizon_returns = pred_samples[:, h] * latest_vol
        
        # Mean forecast for this horizon
        mean_ret = horizon_returns.mean()

        # Empirical confidence interval from simulation
        lower_ret = np.percentile(horizon_returns, 2.5)
        upper_ret = np.percentile(horizon_returns, 97.5)

        # ---------------------------------------------------------
        # HORIZON-BASED UNCERTAINTY SCALING (heuristic)
        # ---------------------------------------------------------
        # Expands uncertainty as forecast horizon increases
        spread_scale = 1 + (h * 0.15)

        # Distance from mean
        lower_dev = mean_ret - lower_ret
        upper_dev = upper_ret - mean_ret

        lower_ret = mean_ret - lower_dev * spread_scale
        upper_ret = mean_ret + upper_dev * spread_scale

        # Convert returns -> prices
        mean_price = last_price * np.exp(mean_ret)
        lower_price = last_price * np.exp(lower_ret)
        upper_price = last_price * np.exp(upper_ret)

        rows.append({
            "date": future_dates[h],
            "forecast_day": h + 1,
            "mean_return": mean_ret,
            "lower_return": lower_ret,
            "upper_return": upper_ret,
            "forecast_price": mean_price,
            "lower_ci": lower_price,
            "upper_ci": upper_price
        })

    # ---------------------------------------------------------
    # RISK METRICS (FINAL HORIZON ONLY)
    # ---------------------------------------------------------
    final_horizon_returns = pred_samples[:, -1]
    var_95 = np.percentile(final_horizon_returns, 5)
    cvar_95 = final_horizon_returns[final_horizon_returns <= var_95].mean()

    print(f"5D VaR(95%): {var_95:.4f}")
    print(f"5D CVaR(95%): {cvar_95:.4f}")

    rows.append({
        "5D VaR(95%)": var_95,
        "5D CVarR(95%)": cvar_95
    }) # We can append on the end of the dataframe as this VaR and CVaR are not for each individual day, only for the end of the five dayss

    return pd.DataFrame(rows)

In [10]:
# =========================================================
# 7) HISTORICAL PLOT (MULTI-HORIZON VERSION)
# =========================================================
def plot_historical_model_fit(
    model,
    scaler,
    df,
    feature_cols,
    lookback=90,
    history_window=250
):
    """
    Evaluates model performance by comparing:
    - predicted 5-day-ahead returns (t+5 horizon only)
    - realized 5-day future price movements

    IMPORTANT:
    This is NOT a full multi-horizon evaluation.
    We intentionally isolate the t+5 prediction because:
    - it is the most stable horizon signal
    - shorter horizons are noisier and harder to interpret visually
    """

    # ---------------------------------------------------------
    # CLEAN EVALUATION DATASET
    # ---------------------------------------------------------
    target_cols = [f"target_{h}d" for h in range(1, 6)]
    eval_df = df.copy()

    # Remove invalid numeric values that could break scaling or sequencing
    eval_df = eval_df.replace([np.inf, -np.inf], np.nan)
    # Safety clipping for extreme market anomalies / bad data points
    eval_df = eval_df.clip(-1e6, 1e6)

    # Ensure full feature + target availability
    eval_df = eval_df.dropna(subset=feature_cols + target_cols).copy()

    # ---------------------------------------------------------
    # FEATURE SCALING (must match training transformation)
    # ---------------------------------------------------------
    features = scaler.transform(eval_df[feature_cols].values)

    # ---------------------------------------------------------
    # BUILD SEQUENCES FOR LSTM INPUT
    # ---------------------------------------------------------
    # Each sample:
    # X[t-lookback:t] -> predicts y[t] (vector of 5 horizons)
    X, y = create_sequences(features, eval_df[target_cols].values, lookback)

    # ---------------------------------------------------------
    # MODEL PREDICTIONS
    # ---------------------------------------------------------
    # Shape: (samples, 5 horizons)
    preds = model.predict(X, verbose=0)

    # ---------------------------------------------------------
    # ISOLATE 5-DAY HORIZON (t+5)
    # ---------------------------------------------------------
    # We focus only on the last horizon:
    # index 4 corresponds to target_5d

    # Convert volatility-scaled returns back to realized scale
    vol = eval_df["volatility_20"].iloc[lookback:].values
    preds_5d = preds[:, 4] * vol

    # ---------------------------------------------------------
    # ALIGN BASE TIMESTAMPS
    # ---------------------------------------------------------
    # These represent the "decision points" (prediction dates)
    base_dates = eval_df.index[lookback:]

    # Shift forward by 5 business days to approximate realization time
    future_dates = base_dates + BDay(5)

    # ---------------------------------------------------------
    # PRICE RECONSTRUCTION FROM LOG RETURNS
    # ---------------------------------------------------------
    # Convert predicted log returns into price space
    base_prices = eval_df["Close"].values[lookback:]
    pred_prices = base_prices * np.exp(preds_5d)

    # ---------------------------------------------------------
    # REALIZED FUTURE PRICES (APPROXIMATION)
    # ---------------------------------------------------------
    # We align actual observed prices at t+5 using nearest available data
    real_prices = (eval_df["Close"].reindex(future_dates, method="nearest").values)

    # ---------------------------------------------------------
    # PLOTTING
    # ---------------------------------------------------------
    plt.figure(figsize=(14, 6))
    plt.plot(future_dates[-history_window:], real_prices[-history_window:], label="Realized 5-Day Future Price")
    plt.plot(future_dates[-history_window:], pred_prices[-history_window:], label="Predicted 5-Day Future Price")
    plt.title("Historical Fit (t+5 Horizon Only)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [11]:
# =========================================================
# 8) FORECAST PLOT (MULTI-HORIZON VERSION)
# =========================================================
def plot_mc_forecast(df, forecast_df, history_window=35):
    """
    Visualizes:

    - Recent historical stock prices
    - 5-step ahead forecast path (t+1 ... t+5)
    - Monte Carlo dropout-based confidence intervals

    IMPORTANT:
    This is a probabilistic forecast visualization, where:
    - forecasts come from a stochastic neural network (MC dropout)
    - uncertainty is estimated empirically from repeated forward passes
    - each horizon is modeled independently (not recursively)
    """
    forecast_df = forecast_df.copy()

    # ---------------------------------------------------------
    # CLEAN FORECAST OUTPUT
    # ---------------------------------------------------------
    # Remove invalid numerical entries that may come from sampling instability
    forecast_df = forecast_df.replace([np.inf, -np.inf], np.nan)
    forecast_df = forecast_df.dropna(
        subset=["date", "forecast_price", "lower_ci", "upper_ci"]
    )

    # ---------------------------------------------------------
    # INITIALIZE FIGURE
    # ---------------------------------------------------------
    plt.figure(figsize=(14, 6))

    # ---------------------------------------------------------
    # HISTORICAL PRICE WINDOW
    # ---------------------------------------------------------
    # We only plot the most recent segment of history for readability
    hist = (df["Close"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .iloc[-history_window:]
    )

    plt.plot(hist.index, hist.values, label="Historical Close")
    # ---------------------------------------------------------
    # FORECAST PATH (MEAN ESTIMATE)
    # ---------------------------------------------------------
    # Each point corresponds to a different forecast horizon (1–5 days ahead)
    plt.plot(forecast_df["date"], forecast_df["forecast_price"], marker="o", label="Forecasted Price Path")

    # ---------------------------------------------------------
    # CONFIDENCE INTERVALS (MC DROPOUT DERIVED)
    # ---------------------------------------------------------
    # These bands reflect:
    # - Monte Carlo sampling uncertainty
    # - additional horizon-dependent widening heuristic
    plt.fill_between(forecast_df["date"], forecast_df["lower_ci"], forecast_df["upper_ci"], alpha=0.25, label="Confidence Band (MC Dropout)")

    # ---------------------------------------------------------
    # VISUAL SEPARATION BETWEEN HISTORY AND FORECAST
    # ---------------------------------------------------------
    plt.axvline(df.index[-1], linestyle="--", alpha=0.7)

    # ---------------------------------------------------------
    # FINAL PLOT STYLING
    # ---------------------------------------------------------
    plt.title("LSTM Multi-Horizon Forecast with MC Dropout Uncertainty")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [12]:
# =========================================================
# DIAGNOSTIC TEST CELL (OPTIONAL EXECUTION)
# =========================================================
# WARNING:
# This cell performs statistical diagnostics (KPSS + ARCH)
# across multiple companies.
#
# It is NOT required for model training or inference.
#
# Running it will generate a large output and may take time.
#
# Running this code will generate KPSS warnings when the test statistic
# falls outside the lookup table used to approximate p-values.
#
# In these cases, the p-value is effectively below the minimum threshold
# available in the table (i.e., strong rejection of stationarity).
#
# This is expected for financial price series and does not affect
# the validity of the conclusion (non-stationarity).
#
# Only execute if you explicitly want to validate:
# - stationarity assumptions
# - heteroskedasticity presence
# across the dataset.
# =========================================================
companies = [
    "Altri",
    "Banco_Comercial_Portugues",
    "Corticeira_Amorim",
    "CTT_Correios_de_Portugal",
    "EDP_Renovaveis",
    "EDP",
    "Galp_Energia",
    "Ibersol",
    "Jeronimo_Martins",
    "Mota_Engil",
    "NOS_SGPS",
    "Novabase",
    "Pharol",
    "REN",
    "Semapa",
    "Sonae",
    "The_Navigator"
]

results = []

for company in companies:

    print(f"Processing {company}...")
    df = pd.read_csv(f"data/{company}_Stock_Price.csv")
    df = clean_stock_data(df)
    price_series = df["Close"].dropna()

    # -----------------------------
    # KPSS TEST (stationarity)
    # H0: stationary
    # H1: non-stationary
    # -----------------------------
    kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
    is_stationary = "Yes" if kpss_pvalue > 0.05 else "No"

    # -----------------------------
    # ARCH TEST (heteroskedasticity)
    # applied on returns
    # -----------------------------
    returns = np.log(price_series).diff().dropna()
    arch_stat, arch_pvalue, _, _ = het_arch(returns)
    has_heteroskedasticity = "Yes" if arch_pvalue < 0.05 else "No"

    # -----------------------------
    # STORE RESULT
    # -----------------------------
    results.append({
        "Company": company,
        "KPSS Statistic": kpss_stat,
        "KPSS p-value": kpss_pvalue,
        "Stationary (KPSS)": is_stationary,
        "ARCH Statistic": arch_stat,
        "ARCH p-value": arch_pvalue,
        "Heteroskedasticity (ARCH)": has_heteroskedasticity
    })

# =========================================================
# FINAL SUMMARY TABLE
# =========================================================
test_df = pd.DataFrame(results)

# Optional: sort for readability
test_df = test_df.sort_values("Company")

display(test_df)

Processing Altri...
Processing Banco_Comercial_Portugues...
Processing Corticeira_Amorim...
Processing CTT_Correios_de_Portugal...
Processing EDP_Renovaveis...
Processing EDP...
Processing Galp_Energia...
Processing Ibersol...
Processing Jeronimo_Martins...


C:\Users\jorge\AppData\Local\Temp\ipykernel_32456\3024064877.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
C:\Users\jorge\AppData\Local\Temp\ipykernel_32456\3024064877.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
C:\Users\jorge\AppData\Local\Temp\ipykernel_32456\3024064877.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
C:\Users\jorge\AppData\Local\Temp\ipykernel_32456

C:\Users\jorge\AppData\Local\Temp\ipykernel_32456\3024064877.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
C:\Users\jorge\AppData\Local\Temp\ipykernel_32456\3024064877.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
C:\Users\jorge\AppData\Local\Temp\ipykernel_32456\3024064877.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
C:\Users\jorge\AppData\Local\Temp\ipykernel_32456

Processing Mota_Engil...
Processing NOS_SGPS...
Processing Novabase...
Processing Pharol...
Processing REN...


Processing Semapa...
Processing Sonae...
Processing The_Navigator...


C:\Users\jorge\AppData\Local\Temp\ipykernel_32456\3024064877.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
C:\Users\jorge\AppData\Local\Temp\ipykernel_32456\3024064877.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
C:\Users\jorge\AppData\Local\Temp\ipykernel_32456\3024064877.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(price_series, regression="c", nlags="auto")
C:\Users\jorge\AppData\Local\Temp\ipykernel_32456

,Company,KPSS Statistic,KPSS p-value,Stationary (KPSS),ARCH Statistic,ARCH p-value,Heteroskedasticity (ARCH)
0,Altri,9.500640,0.01,No,262.110657,1.535881e-50,Yes
1,Banco_Comercial_Portugues,4.459159,0.01,No,283.265572,5.326698e-55,Yes
3,CTT_Correios_de_Portugal,1.593644,0.01,No,22.742593,1.173764e-02,Yes
2,Corticeira_Amorim,8.411787,0.01,No,197.769445,4.709344e-37,Yes
5,EDP,9.441312,0.01,No,461.169508,8.649506e-93,Yes
4,EDP_Renovaveis,7.282151,0.01,No,186.491231,1.049723e-34,Yes
6,Galp_Energia,5.545926,0.01,No,115.811313,3.573364e-20,Yes
7,Ibersol,5.142838,0.01,No,473.998197,1.580286e-95,Yes
8,Jeronimo_Martins,8.615047,0.01,No,35.772459,9.210023e-05,Yes
9,Mota_Engil,2.445795,0.01,No,145.070800,3.841856e-26,Yes


In [ ]:
# =========================================================
# 9) MULTI-STOCK TRAINING + FORECAST PIPELINE
# =========================================================
# This section runs the full end-to-end workflow across
# all selected PSI20 stocks:
#
# Pipeline steps per company:
# 1. Load raw price data
# 2. Train LSTM model (feature engineering included internally)
# 3. Generate probabilistic forecast (MC Dropout)
# 4. Save forecast outputs
# 5. Run diagnostic visualizations
# 6. Print forecast summary
#
# NOTE:
# Each company is trained independently (no shared model).
# This ensures no cross-stock leakage or contamination.
companies = [
    "Altri",
    "Banco_Comercial_Portugues",
    "Corticeira_Amorim",
    "CTT_Correios_de_Portugal",
    "EDP_Renovaveis",
    "EDP",
    "Galp_Energia",
    "Ibersol",
    "Jeronimo_Martins",
    "Mota_Engil",
    "NOS_SGPS",
    "Novabase",
    "Pharol",
    "REN",
    "Semapa",
    "Sonae",
    "The_Navigator"
]
# =========================================================
# MAIN LOOP: COMPANY-BY-COMPANY TRAINING
# =========================================================
for company in companies:
    print("\n" + "=" * 50)
    print(f"Processing: {company}")
    print("=" * 50)
    
    # ---------------------------------------------------------
    # 1) LOAD RAW DATA
    # ---------------------------------------------------------
    # Each file contains OHLCV data for a single stock
    df = pd.read_csv(f"data/{company}_Stock_Price.csv")

    # ---------------------------------------------------------
    # 2) TRAIN MODEL PIPELINE
    # ---------------------------------------------------------
    # Includes:
    # - cleaning
    # - feature engineering
    # - scaling
    # - LSTM training
    model, scaler, df_ready, feature_cols = train_lstm_forecaster(df)

    # ---------------------------------------------------------
    # 3) GENERATE PROBABILISTIC FORECAST
    # ---------------------------------------------------------
    # Uses Monte Carlo Dropout only at inference time
    # to approximate predictive uncertainty
    forecast_df = mc_dropout_forecast_path(model, scaler, df_ready, feature_cols, horizon=5)

    # ---------------------------------------------------------
    # 4) SAVE FORECAST OUTPUTS
    # ---------------------------------------------------------
    # Stores per-company predictions for later benchmarking
    forecast_df.to_csv(f"forecasts/{company}_forecast.csv", index=False)

    # ---------------------------------------------------------
    # 5) DIAGNOSTIC VISUALIZATION (MODEL FIT)
    # ---------------------------------------------------------
    # Compares historical predictions vs realized 5-day outcomes
    plot_historical_model_fit(model, scaler, df_ready, feature_cols)

    # ---------------------------------------------------------
    # 6) FORECAST VISUALIZATION (MC DROPOUT PATH)
    # ---------------------------------------------------------
    # Shows:
    # - predicted price path
    # - confidence intervals
    # - recent history context
    plot_mc_forecast(df_ready, forecast_df)

    # ---------------------------------------------------------
    # 7) OUTPUT SUMMARY
    # ---------------------------------------------------------
    # Quick inspection of forecast structure
    print(forecast_df)

In [12]:
# =========================================================
# TRAIN MODEL (SINGLE FIT PER ASSET)
# =========================================================
def train_model_once(company_df):
    """
    Trains a single LSTM model for one asset.

    Pipeline:
    1. Clean raw OHLCV data
    2. Engineer predictive features
    3. Train LSTM forecaster on full historical dataset

    NOTE:
    This is NOT a walk-forward training procedure.
    The model is trained once per asset using all available data.
    """

    # ---------------------------------------------------------
    # DATA PREPROCESSING
    # ---------------------------------------------------------
    # Clean raw market data (fix formatting, missing values, etc.)
    df = clean_stock_data(company_df)

    # Feature engineering (returns, volatility, trend, volume, etc.)
    df = add_features(df)

    # ---------------------------------------------------------
    # MODEL TRAINING PIPELINE
    # ---------------------------------------------------------
    # Full LSTM training pipeline:
    # - scaling
    # - sequence creation
    # - model fitting
    model, scaler, df_ready, feature_cols = train_lstm_forecaster(df)

    return model, scaler, df_ready, feature_cols

In [13]:
# =========================================================
# WALK-FORWARD INFERENCE (EVALUATION ONLY)
# =========================================================
def walk_forward_inference(
    df,
    model,
    scaler,
    feature_cols,
    lookback=90,
    horizon_index=4
):
    """
    Performs model evaluation on unseen data using a fixed trained model.

    This function does NOT retrain the model.

    Instead it:
    - applies the trained scaler
    - builds sliding-window sequences
    - generates predictions
    - compares predictions vs realized outcomes

    ---------------------------------------------------------
    IMPORTANT CONCEPTS
    ---------------------------------------------------------

    horizon_index:
        Selects which forecast horizon to evaluate:
        - 0 → t+1 day
        - 1 → t+2 days
        - ...
        - 4 → t+5 days (default used in most experiments)

    Volatility scaling:
        Model predicts *standardized returns*
        These are rescaled using realized volatility (vol_20)
        to approximate real return magnitude.
    """

    # ---------------------------------------------------------
    # DEFINE TARGET STRUCTURE
    # ---------------------------------------------------------
    target_cols = [f"target_{h}d" for h in range(1, 6)]

    # ---------------------------------------------------------
    # COPY DATA FOR SAFETY
    # ---------------------------------------------------------
    # Avoid modifying original dataframe during evaluation
    eval_df = df.copy()

    # ---------------------------------------------------------
    # CLEAN INVALID NUMERICAL VALUES
    # ---------------------------------------------------------
    eval_df = eval_df.replace([np.inf, -np.inf], np.nan)

    # Ensure all required inputs exist (features + targets)
    eval_df = eval_df.dropna(subset=feature_cols + target_cols).copy()

    # ---------------------------------------------------------
    # FEATURE SCALING (must match training scaler)
    # ---------------------------------------------------------
    features = scaler.transform(eval_df[feature_cols].values)

    # ---------------------------------------------------------
    # SEQUENCE GENERATION
    # ---------------------------------------------------------
    # Converts time series into supervised learning format:
    # X[t-lookback:t] → predicts y[t]
    X, y = create_sequences(features, eval_df[target_cols].values, lookback)

    # ---------------------------------------------------------
    # MODEL PREDICTION
    # ---------------------------------------------------------
    # Shape:
    # preds → (samples, 5 horizons)
    preds = model.predict(X, verbose=0)

    # ---------------------------------------------------------
    # VOLATILITY RESCALING
    # ---------------------------------------------------------
    # Model outputs standardized returns
    # We rescale using realized rolling volatility
    vol = eval_df["volatility_20"].iloc[lookback:].values

    # ---------------------------------------------------------
    # SELECT HORIZON FOR EVALUATION
    # ---------------------------------------------------------
    # We evaluate only one horizon at a time (e.g. t+5)
    pred_returns = preds[:, horizon_index] * vol
    actual_returns = y[:, horizon_index] * vol

    # ---------------------------------------------------------
    # CONVERT RETURNS TO PRICE SPACE
    # ---------------------------------------------------------
    base_prices = eval_df["Close"].iloc[lookback:].values
    pred_prices = base_prices * np.exp(pred_returns)
    actual_prices = base_prices * np.exp(actual_returns)

    # ---------------------------------------------------------
    # RETURN STRUCTURED RESULTS
    # ---------------------------------------------------------
    return {
        "pred_returns": pred_returns,
        "actual_returns": actual_returns,
        "pred_prices": pred_prices,
        "actual_prices": actual_prices,
        "vols": vol
    }

In [14]:
def classify_return(x, vol, k=0.15):
    """
    Converts a continuous return into a 3-class market regime signal.

    Regimes:
    - bullish: return exceeds positive volatility-adjusted threshold
    - bearish: return falls below negative volatility-adjusted threshold
    - neutral: return lies within the volatility band

    The threshold is adaptive and scales with recent market volatility:
        threshold = k * vol

    This ensures classification sensitivity adjusts to changing market conditions.
    """
    threshold = vol * k

    if x > threshold:
        return "bullish"
    elif x < -threshold:
        return "bearish"
    else:
        return "neutral"

In [15]:
def regime_distance(pred, actual):
    """
    Computes a discrete distance metric between predicted and actual market regimes.

    Each regime is mapped onto an ordinal scale:
        bearish = -1
        neutral = 0
        bullish = 1

    The metric measures the mean absolute distance between predicted and actual regimes.

    Interpretation:
    - 0.0 → perfect classification
    - 1.0 → average error of one regime step (e.g., bullish vs neutral)
    - 2.0 → maximum error (e.g., bullish vs bearish)

    This provides a simple ordinal loss reflecting how "far off" predictions are,
    rather than treating all misclassifications equally.
    """
    mapping = {
        "bearish": -1,
        "neutral": 0,
        "bullish": 1
    }

    p = pred.map(mapping)
    a = actual.map(mapping)

    return np.mean(np.abs(p - a))

In [17]:
# =========================================================
# BACKTESTING LOOP (MULTI-ASSET EVALUATION)
# =========================================================
# This section runs the full evaluation pipeline across all
# selected PSI20 stocks.

backtest_results = {}

companies = [
    "Altri", "Banco_Comercial_Portugues", "Corticeira_Amorim",
    "CTT_Correios_de_Portugal", "EDP_Renovaveis", "EDP",
    "Galp_Energia", "Ibersol", "Jeronimo_Martins", "Mota_Engil",
    "NOS_SGPS", "Novabase", "Pharol", "REN", "Semapa",
    "Sonae", "The_Navigator"
]

for company in companies:
    print("\n" + "=" * 30)
    print(company)
    print("=" * 30)

    # ---------------------------------------------------------
    # LOAD RAW DATA
    # ---------------------------------------------------------
    # Each dataset contains OHLCV time series for a single asset
    df = pd.read_csv(f"data/{company}_Stock_Price.csv")

    # ---------------------------------------------------------
    # TRAIN MODEL (SINGLE ASSET FIT)
    # ---------------------------------------------------------
    # Trains LSTM on full historical dataset for this asset
    model, scaler, df_ready, feature_cols = train_model_once(df)

    # ---------------------------------------------------------
    # WALK-FORWARD INFERENCE (OUT-OF-SAMPLE EVALUATION)
    # ---------------------------------------------------------
    # Evaluates model performance using a fixed trained model.
    # The 5-day forecast horizon is used (index = 4).
    results = walk_forward_inference(
        df_ready,
        model,
        scaler,
        feature_cols,
        horizon_index=4
    )

    # ---------------------------------------------------------
    # EXTRACT PREDICTIONS
    # ---------------------------------------------------------
    pred_returns = results["pred_returns"]
    actual_returns = results["actual_returns"]
    pred_prices = results["pred_prices"]
    actual_prices = results["actual_prices"]
    vols = results["vols"]

    # =========================================================
    # PRICE-SPACE ERROR METRICS
    # =========================================================
    # These metrics evaluate accuracy in reconstructed price space
    price_error = pred_prices - actual_prices
    mae = np.mean(np.abs(price_error))
    rmse = np.sqrt(np.mean(price_error ** 2))

    print("\nPRICE METRICS")
    print("MAE:", mae)
    print("RMSE:", rmse)

    # =========================================================
    # DIRECTIONAL ACCURACY (SIGN PREDICTION)
    # =========================================================
    # Converts continuous returns into directional signals:
    #   +1 → positive return
    #   -1 → negative return
    #    0 → neutral (exact zero return)
    threshold = 0

    pred_dir = np.where(
        pred_returns > threshold,
        1,
        np.where(pred_returns < threshold, -1, 0)
    )

    actual_dir = np.where(
        actual_returns > threshold,
        1,
        np.where(actual_returns < threshold, -1, 0)
    )

    direction_acc = np.mean(pred_dir == actual_dir)

    print("\nDIRECTION METRICS")
    print("Direction Accuracy:", direction_acc)

    # =========================================================
    # REGIME CLASSIFICATION ANALYSIS
    # =========================================================
    print("\nREGIME ANALYSIS")

    # -----------------------------------------------------
    # Convert continuous returns into discrete regimes
    # using volatility-adjusted thresholds
    # -----------------------------------------------------
    pred_class = pd.Series([classify_return(r, v) for r, v in zip(pred_returns, vols)])
    actual_class = pd.Series([classify_return(r, v) for r, v in zip(actual_returns, vols)])

    # -----------------------------------------------------
    # REGIME PERFORMANCE METRICS
    # -----------------------------------------------------
    regime_acc = np.mean(pred_class == actual_class)
    regime_dist = regime_distance(pred_class, actual_class)

    print("Regime Accuracy:", regime_acc)
    print("Regime Distance:", regime_dist)

    print("\nActual Regime Distribution:")
    print(actual_class.value_counts(normalize=True))

    # =========================================================
    # STORE BACKTEST RESULTS
    # =========================================================
    backtest_results[company] = {
        "mae": mae,
        "rmse": rmse,
        "direction_accuracy": direction_acc,
        "regime_accuracy": regime_acc,
        "regime_distance": regime_dist
    }


Altri
Epoch 1/40


C:\dev\Intelligent_Market_Analysis\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - loss: 1.5098 - mae: 1.5098 - val_loss: 1.4321 - val_mae: 1.4321
Epoch 2/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - loss: 1.5020 - mae: 1.5020 - val_loss: 1.4331 - val_mae: 1.4331
Epoch 3/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - loss: 1.4994 - mae: 1.4994 - val_loss: 1.4331 - val_mae: 1.4331
Epoch 4/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 1.4946 - mae: 1.4946 - val_loss: 1.4404 - val_mae: 1.4404
Epoch 5/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - loss: 1.4872 - mae: 1.4872 - val_loss: 1.4421 - val_mae: 1.4421
Epoch 6/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - loss: 1.4790 - mae: 1.4790 - val_loss: 1.4529 - val_mae: 1.4529

PRICE METRICS
MAE: 0.06595901154199786
RMSE: 0.1085362576528448

DIRECTION METRICS
Direction Accuracy: 0.5182911858580899

REGIME ANALYSIS
Regime Accuracy: 0.060888779769211886
Regime Distance: 0.9405843358703658

Actual Regime Distribution:
bullish    0.506506
bearish    0.438006
neutral    0.

C:\dev\Intelligent_Market_Analysis\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - loss: 1.5160 - mae: 1.5160 - val_loss: 1.3812 - val_mae: 1.3812
Epoch 2/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 1.5123 - mae: 1.5123 - val_loss: 1.3890 - val_mae: 1.3890
Epoch 3/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - loss: 1.5081 - mae: 1.5081 - val_loss: 1.3891 - val_mae: 1.3891
Epoch 4/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - loss: 1.5042 - mae: 1.5042 - val_loss: 1.3894 - val_mae: 1.3894
Epoch 5/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - loss: 1.4980 - mae: 1.4980 - val_loss: 1.3827 - val_mae: 1.3827
Epoch 6/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 1.4919 - mae: 1.4919 - val_loss: 1.3791 - val_mae: 1.3791
Epoch 7/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - loss: 1.4878 - mae: 1.4878 - val_loss: 1.3804 - val_mae: 1.3804
Epoch 8/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - loss: 1.4755 - mae: 1.4755 - val_loss: 1.3705 - val_mae: 1.3705
Epoch 9/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/

...

102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - loss: 1.4292 - mae: 1.4292 - val_loss: 1.3894 - val_mae: 1.3894
Epoch 2/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - loss: 1.4228 - mae: 1.4228 - val_loss: 1.3884 - val_mae: 1.3884
Epoch 3/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step - loss: 1.4206 - mae: 1.4206 - val_loss: 1.3907 - val_mae: 1.3907
Epoch 4/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - loss: 1.4196 - mae: 1.4196 - val_loss: 1.3929 - val_mae: 1.3929
Epoch 5/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 103ms/step - loss: 1.4138 - mae: 1.4138 - val_loss: 1.3912 - val_mae: 1.3912
Epoch 6/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 8s 74ms/step - loss: 1.4097 - mae: 1.4097 - val_loss: 1.3871 - val_mae: 1.3871
Epoch 7/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 8s 78ms/step - loss: 1.4004 - mae: 1.4004 - val_loss: 1.3855 - val_mae: 1.3855
Epoch 8/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 60ms/step - loss: 1.3942 - mae: 1.3942 - val_loss: 1.3884 - val_mae: 1.3884
Epoch 9/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - l

C:\dev\Intelligent_Market_Analysis\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - loss: 1.4270 - mae: 1.4270 - val_loss: 1.4479 - val_mae: 1.4479
Epoch 2/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - loss: 1.4224 - mae: 1.4224 - val_loss: 1.4486 - val_mae: 1.4486
Epoch 3/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 1.4179 - mae: 1.4179 - val_loss: 1.4544 - val_mae: 1.4544
Epoch 4/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - loss: 1.4182 - mae: 1.4182 - val_loss: 1.4584 - val_mae: 1.4584
Epoch 5/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - loss: 1.4119 - mae: 1.4119 - val_loss: 1.4800 - val_mae: 1.4800
Epoch 6/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - loss: 1.4094 - mae: 1.4094 - val_loss: 1.4871 - val_mae: 1.4871

PRICE METRICS
MAE: 0.016959427419815987
RMSE: 0.023763558488297278

DIRECTION METRICS
Direction Accuracy: 0.527620918242082

REGIME ANALYSIS
Regime Accuracy: 0.09771667075865456
Regime Distance: 0.9283083722072183

Actual Regime Distribution:
bullish    0.502332
bearish    0.442917
n

C:\dev\Intelligent_Market_Analysis\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


102/102 ━━━━━━━━━━━━━━━━━━━━ 9s 51ms/step - loss: 1.4478 - mae: 1.4478 - val_loss: 1.4119 - val_mae: 1.4119
Epoch 2/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - loss: 1.4412 - mae: 1.4412 - val_loss: 1.4131 - val_mae: 1.4131
Epoch 3/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - loss: 1.4386 - mae: 1.4386 - val_loss: 1.4166 - val_mae: 1.4166
Epoch 4/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - loss: 1.4371 - mae: 1.4371 - val_loss: 1.4166 - val_mae: 1.4166
Epoch 5/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 8s 74ms/step - loss: 1.4358 - mae: 1.4358 - val_loss: 1.4204 - val_mae: 1.4204
Epoch 6/40
102/102 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - loss: 1.4340 - mae: 1.4340 - val_loss: 1.4366 - val_mae: 1.4366

PRICE METRICS
MAE: 0.04923930082403724
RMSE: 0.07091584829576278

DIRECTION METRICS
Direction Accuracy: 0.5550098231827112

REGIME ANALYSIS
Regime Accuracy: 0.14194499017681728
Regime Distance: 0.9243614931237721

Actual Regime Distribution:
bullish    0.529715
bearish    0.417976
neutral    0.0

In [25]:
resultado1 = 0
for company in companies:
    resultado1 += backtest_results[company]['direction_accuracy']
resultado2 = resultado1 / 17
print(resultado2)

0.547221542593647


In [ ]:
backtest_results = {}

companies = ['Banco_Comercial_Portugues', 'CTT_Correios_de_Portugal', 'EDP', 'Galp_Energia', 'Nos_SGPS']

# ---------------------------------------------------------
# REGIME THRESHOLD SENSITIVITY ANALYSIS
# ---------------------------------------------------------
# This experiment evaluates how the regime classification
# threshold (k) affects classification performance.
#
# It is NOT part of the forecasting pipeline.
# It is purely used to select a stable k value for regime labeling.
# ---------------------------------------------------------
k_values = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
for company in companies:
    print("\n==================================")
    print(company)
    print("==================================")

    # ---------------------------------------------------------
    # LOAD DATA
    # ---------------------------------------------------------
    df = pd.read_csv(f"data/{company}_Stock_Price.csv")

    # ---------------------------------------------------------
    # TRAIN MODEL
    # ---------------------------------------------------------
    model, scaler, df_ready, feature_cols = train_model_once(df)

    # ---------------------------------------------------------
    # WALK-FORWARD INFERENCE (BASE EVALUATION SET)
    # ---------------------------------------------------------
    results = walk_forward_inference(
        df_ready,
        model,
        scaler,
        feature_cols,
        horizon_index=4
    )

    pred_returns = results["pred_returns"]
    actual_returns = results["actual_returns"]
    pred_prices = results["pred_prices"]
    actual_prices = results["actual_prices"]
    vols = results["vols"]

    # ---------------------------------------------------------
    # PRICE ERROR METRICS (MODEL PERFORMANCE BASELINE)
    # ---------------------------------------------------------
    price_error = pred_prices - actual_prices

    mae = np.mean(np.abs(price_error))
    rmse = np.sqrt(np.mean(price_error ** 2))

    print("\nPRICE METRICS")
    print("MAE:", mae)
    print("RMSE:", rmse)

    # ---------------------------------------------------------
    # DIRECTIONAL ACCURACY (SIGN-BASED METRIC)
    # ---------------------------------------------------------
    threshold = 0

    pred_dir = np.where(pred_returns > threshold, 1,
                np.where(pred_returns < threshold, -1, 0))

    actual_dir = np.where(actual_returns > threshold, 1,
                  np.where(actual_returns < threshold, -1, 0))

    direction_acc = np.mean(pred_dir == actual_dir)

    print("\nDIRECTION METRICS")
    print("Direction Accuracy:", direction_acc)

    # ---------------------------------------------------------
    # REGIME ANALYSIS ACROSS MULTIPLE k VALUES
    # ---------------------------------------------------------
    print("\nREGIME ANALYSIS (k-SENSITIVITY)")

    for k in k_values:

        # Convert returns into volatility-adjusted regimes
        pred_class = pd.Series([
            classify_return(r, v, k=k)
            for r, v in zip(pred_returns, vols)
        ])

        actual_class = pd.Series([
            classify_return(r, v, k=k)
            for r, v in zip(actual_returns, vols)
        ])

        regime_acc = np.mean(pred_class == actual_class)
        regime_dist = regime_distance(pred_class, actual_class)

        print(f"\n--- k = {k:.2f} ---")
        print("Regime Accuracy:", regime_acc)
        print("Regime Distance:", regime_dist)

        print("\nClass Distribution (Actual):")
        print(actual_class.value_counts(normalize=True))

    # ---------------------------------------------------------
    # STORE BASE METRICS ONLY (NOT k-SCAN RESULTS)
    # ---------------------------------------------------------
    backtest_results[company] = {
        "mae": mae,
        "rmse": rmse,
        "direction_accuracy": direction_acc
    }

In [ ]:
def plot_backtest(company_name, result, max_points=300):
    pred = result["pred"][-max_points:]
    actual = result["actual"][-max_points:]
    df = result["df"]

    plt.figure(figsize=(14,6))

    plt.plot(actual, label="Actual", linewidth=2)
    plt.plot(pred, label="Predicted", linewidth=2)

    plt.title(f"Backtest: {company_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_error(company_name, result):
    error = result["error"]

    plt.figure(figsize=(14,4))
    plt.plot(error, alpha=0.8)
    plt.axhline(0, linestyle="--")

    plt.title(f"Error Over Time: {company_name}")
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
#for company in companies:
#    plot_backtest(company, backtest_results[company])
#    plot_error(company, backtest_results[company])

In [ ]:
EVAL_DATES = [
    "2018-09-05", # Pre-Covid Pandemic
    "2020-09-04", # During Covid Pandemic
    "2021-09-04", # During Covid Pandemic, but after markets got "used" to the pandemic
    "2022-03-05", # During the start of the Russian invasion of Ukraine
    "2024-10-21", # During the start of the bombings of Gaza by Israel
    "2026-04-13" # During the war in the middle east for the straight of hormuz
]

In [ ]:
# =========================================================
# MAP TARGET DATE → NEAREST AVAILABLE TRADING INDEX
# =========================================================
def get_nearest_index(df, target_date):
    """
    Finds the index of the closest available trading day
    to a given evaluation date.

    This is necessary because:
    - eval_dates may include non-trading days (weekends/holidays)
    - financial data is only indexed on business days
    """
    target_date = pd.to_datetime(target_date)
    return df.index.get_indexer([target_date], method="nearest")[0]

In [ ]:
# =========================================================
# REGIME-BASED WALK-FORWARD BACKTEST (FIXED EVAL DATES)
# =========================================================
def run_regime_backtest(
    df,
    eval_dates,
    lookback=90,
    horizon=5
):
    results = []

    # ---------------------------------------------------------
    # DATA PREPARATION (ONE-TIME CLEANING)
    # ---------------------------------------------------------
    # This ensures all evaluation runs share identical preprocessing
    # and avoids leakage between evaluation points.
    # ---------------------------------------------------------
    df = clean_stock_data(df)
    df = add_features(df)

    feature_cols = [
        "ret_1",
        "ret_5",
        "momentum_10",
        "volatility_5",
        "volatility_20",
        "range_pct",
        "trend_ratio",
        "price_vs_ma20",
        "zscore_20",
        "atr_14",
        "volume_shock",
        "volume_momentum"
    ]
    target_cols = [f"target_{h}d" for h in range(1, 6)]

    # ---------------------------------------------------------
    # CLEAN INVALID VALUES
    # ---------------------------------------------------------
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=feature_cols + target_cols).copy()

    # =========================================================
    # WALK-FORWARD LOOP OVER FIXED EVALUATION DATES
    # =========================================================
    for date in eval_dates:
        # -----------------------------------------------------
        # ALIGN EVALUATION DATE TO NEAREST TRADING DAY
        # -----------------------------------------------------
        idx = get_nearest_index(df, date)

        # -----------------------------------------------------
        # ENSURE SUFFICIENT TRAINING HISTORY
        # -----------------------------------------------------
        if idx < 300:
            continue

        # -----------------------------------------------------
        # ENSURE FUTURE TARGET WINDOW EXISTS
        # -----------------------------------------------------
        if idx + horizon >= len(df):
            continue

        print(f"\nBacktesting at {date}")

        # -----------------------------------------------------
        # TRAIN MODEL USING ONLY DATA UP TO CURRENT DATE
        # -----------------------------------------------------
        train_df = df.iloc[:idx].copy()
        model, scaler, _, feature_cols = train_lstm_forecaster(
            train_df,
            lookback=lookback,
            horizon=horizon
        )

        # -----------------------------------------------------
        # BUILD LAST OBSERVED INPUT WINDOW
        # -----------------------------------------------------
        latest_features = (
            train_df[feature_cols]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .values[-lookback:]
        )

        if len(latest_features) < lookback:
            continue

        latest_scaled = scaler.transform(latest_features)
        X_input = latest_scaled.reshape(1, lookback, len(feature_cols))

        # -----------------------------------------------------
        # MODEL FORECAST (MULTI-HORIZON OUTPUT)
        # -----------------------------------------------------
        # pred_returns shape: (5,)
        # representing t+1 ... t+5 forecasts
        # -----------------------------------------------------
        pred_returns = model.predict(X_input, verbose=0)[0]

        # -----------------------------------------------------
        # REALIZED FUTURE RETURNS (GROUND TRUTH)
        # -----------------------------------------------------
        actual_returns = df.iloc[idx][target_cols].values

        # -----------------------------------------------------
        # VOLATILITY AT TIME OF FORECAST
        # -----------------------------------------------------
        # Used to contextualize regime magnitude
        # -----------------------------------------------------
        vol = df.iloc[idx]["volatility_20"]

        # -----------------------------------------------------
        # STORE RESULTS FOR ALL HORIZONS
        # -----------------------------------------------------
        for h in range(horizon):
            results.append({
                "eval_date": pd.to_datetime(date),
                "horizon": h + 1,
                "pred_return": float(pred_returns[h]),
                "actual_return": float(actual_returns[h]),
                "volatility": float(vol)
            })

    return pd.DataFrame(results)

In [ ]:
# =========================================================
# REGIME BACKTEST VISUALIZATION
# =========================================================
def plot_regime_results(results, company):
    """
    Visualizes walk-forward backtest results for each evaluation date.

    For each evaluation point, the function plots:
    - predicted multi-horizon returns
    - actual realized multi-horizon returns

    This provides a per-regime snapshot of model performance
    across the 1–5 day forecast horizon.
    """
    # ---------------------------------------------------------
    # LOOP OVER ALL EVALUATION DATES
    # ---------------------------------------------------------
    for date in results["eval_date"].unique():
        subset = results[results["eval_date"] == date]

        # -----------------------------------------------------
        # CREATE FIGURE FOR THIS EVALUATION POINT
        # -----------------------------------------------------
        plt.figure(figsize=(10, 5))

        # -----------------------------------------------------
        # ACTUAL FUTURE RETURNS (GROUND TRUTH)
        # -----------------------------------------------------
        plt.plot(subset["horizon"], subset["actual_return"], marker="o", linewidth=2, label="Actual Returns")

        # -----------------------------------------------------
        # MODEL PREDICTIONS
        # -----------------------------------------------------
        plt.plot(subset["horizon"], subset["pred_return"], marker="o", linewidth=2, label="Predicted Returns")

        # -----------------------------------------------------
        # PLOT FORMATTING
        # -----------------------------------------------------
        plt.title(f"{company} — Walk-Forward Backtest ({date})")
        plt.xlabel("Forecast Horizon (Days Ahead)")
        plt.ylabel("Log Return")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
# =========================================================
# WALK-FORWARD BACKTEST EXECUTION (MULTI-COMPANY)
# =========================================================
# This block runs the full evaluation pipeline across all assets.
#
# For each company:
# 1. Load raw price data
# 2. Run fixed-date walk-forward backtest
# 3. Compute error + directional + regime metrics
# 4. Store results for later comparison
# 5. Generate diagnostic plots
# =========================================================
all_results = {}
companies = [
    "Altri",
    "Banco_Comercial_Portugues",
    "Corticeira_Amorim",
    "CTT_Correios_de_Portugal",
    "EDP_Renovaveis",
    "EDP",
    "Galp_Energia",
    "Ibersol",
    "Jeronimo_Martins",
    "Mota_Engil",
    "NOS_SGPS",
    "Novabase",
    "Pharol",
    "REN",
    "Semapa",
    "Sonae",
    "The_Navigator"
]

companies = ['Banco_Comercial_Portugues', 'CTT_Correios_de_Portugal', 'EDP', 'Galp_Energia', 'Nos_SGPS']

for company in companies:
    print("\n====================")
    print(company)
    print("====================")

    # ---------------------------------------------------------
    # LOAD RAW MARKET DATA
    # ---------------------------------------------------------
    df = pd.read_csv(f"data/{company}_Stock_Price.csv")

    # ---------------------------------------------------------
    # RUN WALK-FORWARD REGIME BACKTEST
    # ---------------------------------------------------------
    # This evaluates model performance at predefined evaluation dates (EVAL_DATES),
    # retraining the model up to each evaluation point.
    # ---------------------------------------------------------
    results = run_regime_backtest(df, EVAL_DATES)

    all_results[company] = results

    # =========================================================
    # CONTINUOUS FORECAST ERROR METRICS
    # =========================================================

    error = results["pred_return"] - results["actual_return"]
    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error ** 2))

    print("\nRETURN METRICS")
    print("MAE:", mae)
    print("RMSE:", rmse)

    # =========================================================
    # DIRECTIONAL ACCURACY (SIGN PREDICTION)
    # =========================================================
    # Evaluates whether the model correctly predicts:
    # - positive vs negative return direction
    # ---------------------------------------------------------
    pred_dir = np.sign(results["pred_return"])
    actual_dir = np.sign(results["actual_return"])
    direction_acc = np.mean(pred_dir == actual_dir)

    print("\nDIRECTION METRICS")
    print("Direction Accuracy:", direction_acc)

    # =========================================================
    # REGIME CLASSIFICATION METRICS (VOLATILITY-SCALED)
    # =========================================================
    # Converts continuous returns into:
    # - bullish / neutral / bearish regimes
    # using volatility-adjusted thresholds.
    # =========================================================

    pred_class = pd.Series([
        classify_return(r, v)
        for r, v in zip(
            results["pred_return"],
            results["volatility"]
        )
    ])

    actual_class = pd.Series([
        classify_return(r, v)
        for r, v in zip(
            results["actual_return"],
            results["volatility"]
        )
    ])

    regime_acc = np.mean(pred_class == actual_class)
    regime_dist = regime_distance(pred_class, actual_class)

    print("\nREGIME METRICS")
    print("Regime Accuracy:", regime_acc)
    print("Regime Distance:", regime_dist)

    # =========================================================
    # DIAGNOSTIC VISUALIZATION
    # =========================================================
    # Generates per-evaluation-date plots comparing:
    # - predicted vs actual multi-horizon returns
    # =========================================================
    plot_regime_results(results,company)